In [227]:
!pip install nltk
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer
import re

Defaulting to user installation because normal site-packages is not writeable

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [228]:
critical_keywords = [
    "short circuit", "fire", "spark", "electric shock",
    "water leakage", "gas leak", "explosion", "danger",
    "live wire", "current", "burning smell"
]

In [229]:
sia = SentimentIntensityAnalyzer()

In [230]:
def extract_days(text):
    match = re.search(r'(\d+)\s*day', text.lower())
    if match:
        return int(match.group(1))
    return 0

In [231]:
from nltk.sentiment import SentimentIntensityAnalyzer
import re

sia = SentimentIntensityAnalyzer()

# ✅ Helper: extract days (English)
def extract_days(text):
    text = text.lower()
    match = re.search(r'(\d+)\s*day', text)
    if match:
        return int(match.group(1))
    return 0


# 🚀 FINAL FUNCTION
def predict_urgency(text):
    text_lower = text.lower()

    # 🚨 0. SEMANTIC HAZARD DETECTION (NEW - VERY IMPORTANT)
    hazard_patterns = [
        "hot", "heating", "overheating",
        "burning", "burning smell", "burning odor",
        "smell something burning",
        "warm", "feels warm",
        "melting", "melt",
        "spark", "shock",
        "gas smell", "gas leak",
        "smoke",
        "damaged wiring", "damaged wire"
    ]

    if any(word in text_lower for word in hazard_patterns):
        return "High"
    
    # 🚨 1. HAZARD DETECTION (TOP PRIORITY)
    critical_keywords = [
        "short circuit", "fire", "spark", "electric shock",
        "gas leak", "gas leaking", "leaking gas", "gas is leaking",
        "burning smell", "smoke", "overheating", "overheat",
        "wire overheating", "strange smell", "electrical smell",
        "leak from cylinder", "gas smell",
        
        # 🔥 FINAL FIXES
        "water leakage", "leakage",
        "cable hot", "getting hot"
    ]
    
    if any(word in text_lower for word in critical_keywords):
        return "High"
    
    # ⏳ 2. STRONG DELAY / ESCALATION
    delay_keywords = [
        "no action", "no response", "not resolved",
        "still pending", "multiple times", "repeated complaints"
    ]
    
    if any(word in text_lower for word in delay_keywords):
        return "High"
    
    # ⏳ 3. TIME-BASED LOGIC
    days = extract_days(text)
    
    if days >= 3:
        return "High"
    elif days == 2:
        return "Medium"
    
    # ⚠️ 4. FREQUENCY → MEDIUM
    if "frequent" in text_lower or "again and again" in text_lower:
        return "Medium"
    
    # 🧊 5. LOW PRIORITY (SOFT LANGUAGE)
    low_keywords = [
        "whenever possible", "not urgent",
        "just informing", "minor issue",
        "working fine", "no urgency",
        "just checking", "still usable",
        "manageable", "can be checked later",
        "no hurry", "at your convenience",
        
        # 🔥 FINAL FIXES
        "does not require immediate attention",
        "no immediate attention needed",
        "still working", "working but"
    ]
    
    # Special handling: "not urgent"
    if "not urgent" in text_lower:
        if "frequent" in text_lower or "needs attention" in text_lower:
            return "Medium"
        return "Low"
    
    if any(word in text_lower for word in low_keywords):
        return "Low"
    
    # 🧠 6. NORMAL ISSUES → MEDIUM (DEFAULT)
    normal_issue_keywords = [
        "not working", "issue", "problem",
        "low pressure", "slow", "dim",
        "not working properly"
    ]
    
    if any(word in text_lower for word in normal_issue_keywords):
        return "Medium"
    
    # 😊 7. SENTIMENT FALLBACK (LAST STEP)
    score = sia.polarity_scores(text)
    compound = score['compound']
    
    if compound <= -0.6:
        return "High"
    elif compound >= 0.5:
        return "Low"
    else:
        return "Medium"

In [232]:
class UrgencyAgent:
    def __init__(self):
        self.sia = SentimentIntensityAnalyzer()
    
    def extract_days(self, text):
        match = re.search(r'(\d+)\s*day', text.lower())
        if match:
            return int(match.group(1))
        return 0
    
    def predict(self, text):
        text_lower = text.lower()
        
        urgent_keywords = [
            "urgent", "immediately", "asap", "not working", 
            "no response", "complaint not resolved", "emergency"
        ]
        
        # Rule-based
        if any(word in text_lower for word in urgent_keywords):
            return "High"
        
        # Time-based
        days = self.extract_days(text)
        if days >= 3:
            return "High"
        elif days == 2:
            return "Medium"
        
        # Sentiment-based
        score = self.sia.polarity_scores(text)
        compound = score['compound']
        
        if compound <= -0.05:
            return "High"
        elif compound >= 0.05:
            return "Low"
        else:
            return "Medium"

In [233]:
agent = UrgencyAgent()

complaint = "Water supply has not been available for 4 days"

print("Complaint:", complaint)
print("Predicted Urgency:", agent.predict(complaint))

Complaint: Water supply has not been available for 4 days
Predicted Urgency: High


In [234]:
print(predict_urgency("This is not urgent, small issue"))

Low


In [235]:
text = "There is a short circuit due to water leakage"

print(predict_urgency(text))

High


In [236]:
text = "Road is damaged please fix it"

print(predict_urgency(text))

Medium


In [237]:
text = "Gas leak smell coming from pipeline"

print(predict_urgency(text))

High


In [238]:
text = "thoda spark ho raha hai but manageable"

print(predict_urgency(text))

High


In [239]:
test_data = [
    # 🚨 Hazard cases (should ALWAYS be HIGH)
    ("Short circuit due to water leakage in my house", "High"),
    ("Gas leak smell coming from pipeline", "High"),
    ("Fire broke out in electric meter", "High"),
    
    # ⏳ Time-based cases
    ("Water supply not available for 5 days", "High"),
    ("Electricity issue for 3 days", "High"),
    ("Internet not working for 2 days", "Medium"),
    
    # ⚡ Urgent keyword cases
    ("This issue is very urgent please resolve immediately", "High"),
    ("No response from department, very urgent", "High"),
    
    # 😡 Sentiment-based cases
    ("I am extremely frustrated with this service", "High"),
    ("Very bad experience but not urgent", "Medium"),
    
    # 😐 Normal / neutral cases
    ("Street light not working since yesterday", "Medium"),
    ("Garbage not collected", "Medium"),
    
    # 🙂 Low urgency cases
    ("Please fix the road when possible", "Low"),
    ("Just a small issue, no hurry", "Low"),
    
    # 🧠 Edge cases (important!)
    ("Water leakage but no damage yet", "High"),   # still risky
    ("Minor spark seen once", "High"),             # safety issue
    ("Everything is fine just checking", "Low")
]

In [240]:
correct = 0

for text, expected in test_data:
    predicted = predict_urgency(text)
    
    print("📝 Complaint:", text)
    print("✅ Expected:", expected)
    print("🤖 Predicted:", predicted)
    
    if predicted == expected:
        print("✔️ Correct")
        correct += 1
    else:
        print("❌ Wrong")
    
    print("-" * 60)

accuracy = correct / len(test_data)
print(f"🎯 Accuracy: {accuracy * 100:.2f}%")

📝 Complaint: Short circuit due to water leakage in my house
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Gas leak smell coming from pipeline
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Fire broke out in electric meter
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Water supply not available for 5 days
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Electricity issue for 3 days
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Internet not working for 2 days
✅ Expected: Medium
🤖 Predicted: Medium
✔️ Correct
------------------------------------------------------------
📝 Complaint: This issue is very urgent please resolve immedia

In [241]:
english_unseen_data = [
    # 🚨 Clear hazards
    ("There is a burning smell coming from the switchboard", "High"),
    ("Gas is leaking from the cylinder", "High"),
    ("Electric shock when touching the switch", "High"),
    ("Smoke coming out of the meter box", "High"),
    
    # ⏳ Time-based
    ("Electricity has not been working for 4 days", "High"),
    ("Water supply is unavailable for 3 days", "High"),
    ("Internet has not been working for 2 days", "Medium"),
    ("The issue started yesterday", "Medium"),
    
    # 😡 Frustration / escalation
    ("I have complained multiple times but no action has been taken", "High"),
    ("This issue is still not resolved despite repeated complaints", "High"),
    
    # 😐 Normal issues
    ("Street lights are not working since last night", "Medium"),
    ("Water pressure is very low", "Medium"),
    ("The fan is not working properly", "Medium"),
    
    # 🙂 Low priority (soft language)
    ("Please fix the road whenever possible", "Low"),
    ("This is not urgent, you can check it later", "Low"),
    ("Just informing you about a minor issue", "Low"),
    ("Everything is working fine, just wanted to confirm", "Low"),
    
    # 🧠 Mixed tricky cases
    ("There is a gas smell but it is very mild", "High"),
    ("Water leakage is present but manageable for now", "High"),
    ("There is a small issue but no urgency", "Low"),
    
    # ⚡ Confusing wording
    ("The system is slow but still usable", "Low"),
    ("The issue is not critical but needs attention", "Medium"),
    ("This is not urgent but happens frequently", "Medium"),
    
    # 🧪 Edge cases
    ("Everything seems fine, just checking", "Low"),
    ("No major issue, just informing", "Low"),
    ("The light is slightly dim but working", "Low"),
    
    # 🔥 Hidden danger wording
    ("The wire is overheating", "High"),
    ("There is a strange smell from the electrical panel", "High"),
]

In [242]:
correct = 0

for text, expected in english_unseen_data:
    predicted = predict_urgency(text)
    
    print("📝 Complaint:", text)
    print("✅ Expected:", expected)
    print("🤖 Predicted:", predicted)
    
    if predicted == expected:
        print("✔️ Correct")
        correct += 1
    else:
        print("❌ Wrong")
    
    print("-" * 60)

accuracy = correct / len(english_unseen_data)
print(f"🎯 English Unseen Accuracy: {accuracy * 100:.2f}%")

📝 Complaint: There is a burning smell coming from the switchboard
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Gas is leaking from the cylinder
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Electric shock when touching the switch
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Smoke coming out of the meter box
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Electricity has not been working for 4 days
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Water supply is unavailable for 3 days
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Internet has not been wo

In [243]:
english_unseen_v4 = [
    # 🚨 Clear hazards
    ("There is a strong gas leak in the kitchen", "High"),
    ("The electrical panel is emitting smoke", "High"),
    ("I got an electric shock while using the switch", "High"),
    ("The wire is overheating continuously", "High"),
    
    # ⏳ Time-based issues
    ("The electricity has been out for 5 days", "High"),
    ("Water supply has not been available for 3 days", "High"),
    ("Internet has not been working for 2 days", "Medium"),
    ("The issue started yesterday evening", "Medium"),
    
    # 😡 Escalation / repeated complaints
    ("I have complained several times but no action has been taken", "High"),
    ("This problem is still pending after multiple complaints", "High"),
    
    # 😐 Regular issues
    ("Street lights are not functioning since last night", "Medium"),
    ("Water pressure is very low in my area", "Medium"),
    ("The fan is not working properly", "Medium"),
    
    # 🙂 Low priority
    ("Please repair the road at your convenience", "Low"),
    ("This is not urgent, it can be fixed later", "Low"),
    ("Just informing you about a minor inconvenience", "Low"),
    ("Everything is working fine, just wanted to confirm once", "Low"),
    
    # 🧠 Mixed tricky cases
    ("There is a slight gas smell but it is not very strong", "High"),
    ("Water leakage is present but manageable for now", "High"),
    ("This is a small issue and does not require immediate attention", "Low"),
    
    # ⚖️ Conflicting statements
    ("This is not urgent but happens frequently", "Medium"),
    ("The issue is minor but occurs again and again", "Medium"),
    
    # 🧪 Edge realistic cases
    ("Everything seems fine, just checking", "Low"),
    ("No major problem, just informing for awareness", "Low"),
    ("The light is slightly dim but still working", "Low"),
    
    # 🔥 Hidden hazards
    ("There is a strange burning smell from the switchboard", "High"),
    ("The cable is getting very hot while in use", "High"),
    
    # 🧠 Neutral but important
    ("The system is slow but still usable", "Low"),
    ("The issue is not critical but needs attention", "Medium"),
]

In [244]:
correct = 0

for text, expected in english_unseen_v4:
    predicted = predict_urgency(text)
    
    print("📝 Complaint:", text)
    print("✅ Expected:", expected)
    print("🤖 Predicted:", predicted)
    
    if predicted == expected:
        print("✔️ Correct")
        correct += 1
    else:
        print("❌ Wrong")
    
    print("-" * 60)

accuracy = correct / len(english_unseen_v4)
print(f"🎯 Final English Unseen Accuracy (V4): {accuracy * 100:.2f}%")

📝 Complaint: There is a strong gas leak in the kitchen
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: The electrical panel is emitting smoke
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: I got an electric shock while using the switch
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: The wire is overheating continuously
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: The electricity has been out for 5 days
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Water supply has not been available for 3 days
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Internet has no

In [245]:
english_unseen_v5 = [
    # 🚨 Hazard cases
    ("There is a gas leak near the stove", "High"),
    ("The switchboard is producing a burning smell", "High"),
    ("I experienced an electric shock from the plug", "High"),
    ("The cable is overheating during use", "High"),
    
    # ⏳ Time-based issues
    ("Electricity has been unavailable for 4 days", "High"),
    ("Water supply has been disrupted for 3 days", "High"),
    ("The internet has not worked for 2 days", "Medium"),
    ("This issue started yesterday morning", "Medium"),
    
    # 😡 Escalation
    ("I have reported this issue several times but nothing has been done", "High"),
    ("This complaint is still unresolved after many follow-ups", "High"),
    
    # 😐 Normal issues
    ("Street lights are not working properly", "Medium"),
    ("Water pressure is low throughout the building", "Medium"),
    ("The fan is making noise and not functioning correctly", "Medium"),
    
    # 🙂 Low priority
    ("Please fix this issue whenever you are available", "Low"),
    ("This is not urgent and can be handled later", "Low"),
    ("Just informing you about a small issue", "Low"),
    ("Everything is fine, just wanted to double check", "Low"),
    
    # 🧠 Mixed tricky
    ("There is a mild gas smell but it is not very strong", "High"),
    ("Water leakage exists but can be managed for now", "High"),
    ("This issue is minor and does not need immediate action", "Low"),
    
    # ⚖️ Conflicting intent
    ("This is not urgent but keeps happening frequently", "Medium"),
    ("The issue is small but occurs repeatedly", "Medium"),
    
    # 🧪 Edge cases
    ("Everything appears normal, just checking", "Low"),
    ("No major issue, only informing", "Low"),
    ("The light is dim but still working properly", "Low"),
    
    # 🔥 Hidden hazards
    ("There is a strange smell coming from the wiring", "High"),
    ("The device is getting unusually hot while operating", "High"),
    
    # 🧠 Neutral tricky
    ("The system is slow but still usable", "Low"),
    ("The issue is not serious but requires attention", "Medium"),
]

In [246]:
correct = 0

for text, expected in english_unseen_v5:
    predicted = predict_urgency(text)
    
    print("📝 Complaint:", text)
    print("✅ Expected:", expected)
    print("🤖 Predicted:", predicted)
    
    if predicted == expected:
        print("✔️ Correct")
        correct += 1
    else:
        print("❌ Wrong")
    
    print("-" * 60)

accuracy = correct / len(english_unseen_v5)
print(f"🎯 Final Accuracy (V5): {accuracy * 100:.2f}%")

📝 Complaint: There is a gas leak near the stove
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: The switchboard is producing a burning smell
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: I experienced an electric shock from the plug
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: The cable is overheating during use
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Electricity has been unavailable for 4 days
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Water supply has been disrupted for 3 days
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: The internet has n

In [247]:
hazard_test_data = [
    # 🔥 Direct hazards
    ("There is a gas leak in my kitchen", "High"),
    ("Smoke is coming from the electrical panel", "High"),
    ("I got an electric shock while plugging in a device", "High"),
    ("The wire is overheating rapidly", "High"),
    
    # ⚠️ Indirect hazard wording
    ("There is a strange smell coming from the switchboard", "High"),
    ("The cable feels extremely hot when in use", "High"),
    ("There is a burning odor near the socket", "High"),
    ("The device is heating up unusually", "High"),
    
    # 🧠 Soft but dangerous
    ("There is a slight gas smell in the room", "High"),
    ("I noticed a minor spark from the wire", "High"),
    ("The switch produces a small spark sometimes", "High"),
    ("There is a faint burning smell from the appliance", "High"),
    
    # ❗ Hidden danger
    ("The fuse box is making unusual sounds and smells", "High"),
    ("The plug point feels warm even when not in use", "High"),
    ("The wiring seems damaged and slightly heated", "High"),
    
    # ⚡ Confusing phrasing
    ("There is no visible fire but a burning smell is present", "High"),
    ("It does not look serious but there is a gas smell", "High"),
    ("Everything looks fine but I can smell something burning", "High"),
    
    # 🧪 Edge realistic
    ("The appliance is working but getting very hot", "High"),
    ("The system runs but there is overheating in the panel", "High"),
    ("The wire insulation appears to be melting", "High"),
    
    # 🔄 Tricky negation
    ("Not sure if it is dangerous but there is a gas smell", "High"),
    ("It might not be urgent but the wire is overheating", "High"),
]

In [248]:
correct = 0

for text, expected in hazard_test_data:
    predicted = predict_urgency(text)
    
    print("📝 Complaint:", text)
    print("✅ Expected:", expected)
    print("🤖 Predicted:", predicted)
    
    if predicted == expected:
        print("✔️ Correct")
        correct += 1
    else:
        print("❌ Wrong")
    
    print("-" * 60)

accuracy = correct / len(hazard_test_data)
print(f"🚨 Hazard Detection Accuracy: {accuracy * 100:.2f}%")

📝 Complaint: There is a gas leak in my kitchen
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: Smoke is coming from the electrical panel
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: I got an electric shock while plugging in a device
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: The wire is overheating rapidly
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: There is a strange smell coming from the switchboard
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: The cable feels extremely hot when in use
✅ Expected: High
🤖 Predicted: High
✔️ Correct
------------------------------------------------------------
📝 Complaint: There is a bu